In [ ]:
import pandas as pd

# -----------------------------
# LOAD WFO DATA
# -----------------------------
file_path = "/content/name.tsv"   # update path

df = pd.read_csv(file_path, sep="\t", low_memory=False)

print("Total rows:", len(df))

Total rows: 1657866


In [ ]:
# The column 'taxonRank' was not found in the DataFrame 'df', which caused the KeyError.
# Please inspect the available columns in 'df' to find the correct column name for taxonomic rank.
print("Columns available in the DataFrame 'df':")
print(df.columns)

# Original line (commented out to prevent error):
# df_species = df[df["taxonRank"] == "species"]

Columns available in the DataFrame 'df':
Index(['ID', 'alternativeID', 'basionymID', 'scientificName', 'authorship',
       'rank', 'uninomial', 'genus', 'infragenericEpithet', 'specificEpithet',
       'infraspecificEpithet', 'code', 'referenceID', 'publishedInYear',
       'link'],
      dtype='object')


In [ ]:
df_species = df[df["rank"] == "species"]

In [ ]:
import pandas as pd

df = pd.read_csv("taxon.tsv", sep="\t", low_memory=False)

print(df.columns)
print("Total rows:", len(df))

Index(['ID', 'nameID', 'parentID', 'accordingToID', 'scrutinizer',
       'scrutinizerID', 'scrutinizerDate', 'referenceID', 'extinct', 'link'],
      dtype='object')
Total rows: 453167


In [ ]:
import pandas as pd

names = pd.read_csv("name.tsv", sep="\t", low_memory=False)
taxon = pd.read_csv("taxon.tsv", sep="\t", low_memory=False)

print(names.columns)
print(taxon.columns)

Index(['ID', 'alternativeID', 'basionymID', 'scientificName', 'authorship',
       'rank', 'uninomial', 'genus', 'infragenericEpithet', 'specificEpithet',
       'infraspecificEpithet', 'code', 'referenceID', 'publishedInYear',
       'link'],
      dtype='object')
Index(['ID', 'nameID', 'parentID', 'accordingToID', 'scrutinizer',
       'scrutinizerID', 'scrutinizerDate', 'referenceID', 'extinct', 'link'],
      dtype='object')


In [ ]:
genera = [
"Sambucus","Viburnum","Liquidambar","Anacardium","Mangifera",
"Pistacia","Rhus","Toxicodendron","Pachypodium","Ilex",
"Schefflera","Alnus","Betula","Carpinus","Corylus",
"Catalpa","Jacaranda","Tabebuia","Celtis","Cornus",
"Dipterocarpus","Shorea","Diospyros","Arbutus","Acacia",
"Bauhinia","Caesalpinia","Gleditsia","Laburnum","Robinia",
"Castanea","Fagus","Lithocarpus","Quercus","Fouquieria",
"Carya","Juglans","Pterocarya","Bertholletia","Lagerstroemia",
"Adansonia","Bombax","Brachychiton","Ceiba","Durio",
"Theobroma","Tilia","Azadirachta","Melia","Swietenia",
"Ficus","Morus","Eucalyptus","Eugenia","Myrtus",
"Psidium","Nothofagus","Davidia","Nyssa","Fraxinus",
"Olea","Paulownia","Platanus","Rhizophora","Crataegus",
"Malus","Prunus","Pyrus","Sorbus","Coffea",
"Citrus","Phellodendron","Tetradium","Populus","Salix",
"Acer","Aesculus","Koelreuteria","Litchi","Ungnadia",
"Argania","Palaquium","Sideroxylon","Ailanthus","Stewartia",
"Ulmus","Zelkova","Cordyline","Dracaena","Yucca",
"Areca","Phoenix","Trachycarpus","Annona","Asimina",
"Cinnamomum","Laurus","Persea","Sassafras","Liriodendron",
"Magnolia","Myristica"
]

df_filtered = df_species[df_species["genus"].isin(genera)]

In [ ]:
df = names.merge(taxon, left_on="ID", right_on="nameID", how="inner")

print("Merged rows:", len(df))

Merged rows: 453167


In [ ]:
df_species = df[df["rank"] == "species"]

In [ ]:
df_species = df_species[df_species["genus"].isin(genera)]

In [ ]:
df_species = df_species.drop_duplicates(subset=["scientificName"])

In [ ]:
df_species.to_csv("wfo_species_clean.csv", index=False)

print("Final species count:", len(df_species))

Final species count: 12087


In [ ]:
df_species["genus"].value_counts().head(50)

,count
genus,
Eugenia,1262
Acacia,1112
Ficus,873
Eucalyptus,861
Diospyros,804
Quercus,628
Salix,617
Ilex,578
Prunus,389


In [ ]:
df_species["scientificName"].nunique()

12087

In [ ]:
df_species["scientificName"].duplicated().sum()

np.int64(0)

In [ ]:
df_species = df_species[df_species["rank"] == "species"]

In [ ]:
df_species["clean"] = df_species["scientificName"].str.replace(r"\s+.*$", "", regex=True)
df_species = df_species.drop_duplicates(subset=["clean"])

In [ ]:
df_species = df_species[df_species["scientificName"].str.count(" ") == 1]

In [ ]:
import requests
import pandas as pd
import time

# load your species
df = pd.read_csv("wfo_species_clean.csv")
species_list = df["scientificName"].dropna().unique()


def get_wikipedia_summary(species):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{species.replace(' ', '_')}"

    try:
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None
        return r.json().get("extract")
    except:
        return None


def extract_traits(text):
    if not text:
        return {}

    text_lower = text.lower()

    traits = {}

    # bloom time
    if "flower" in text_lower:
        traits["has_flowering_info"] = True

    # native range
    if "native to" in text_lower:
        start = text_lower.find("native to")
        traits["native_range"] = text[start:start+100]

    # climate
    if "tropical" in text_lower:
        traits["climate"] = "tropical"
    elif "temperate" in text_lower:
        traits["climate"] = "temperate"

    return traits


results = []

for i, sp in enumerate(species_list[:500]):  # test first

    text = get_wikipedia_summary(sp)
    traits = extract_traits(text)

    results.append({
        "scientificName": sp,
        "summary": text,
        **traits
    })

    if i % 50 == 0:
        print(f"Processed {i}")

    time.sleep(0.05)


df_traits = pd.DataFrame(results)
df_traits.to_csv("species_traits.csv", index=False)

print("DONE")

Processed 0
Processed 50
Processed 100
Processed 150
Processed 200
Processed 250
Processed 300
Processed 350
Processed 400
Processed 450
DONE


In [ ]:
import pandas as pd

# load TRY dataset
try_df = pd.read_csv("tree_trait_matrix.csv")

# load your WFO species
wfo_df = pd.read_csv("wfo_species_clean.csv")

# standardize names (CRITICAL)
def clean_name(x):
    if pd.isna(x):
        return x
    return x.strip().lower()

try_df["SpeciesName"] = try_df["SpeciesName"].apply(clean_name)
wfo_df["scientificName"] = wfo_df["scientificName"].apply(clean_name)

# keep only your 12k species
species_set = set(wfo_df["scientificName"])

try_df = try_df[try_df["SpeciesName"].isin(species_set)]

print("TRY species after filter:", try_df["SpeciesName"].nunique())

TRY species after filter: 7013


In [ ]:
selected_traits = [
    "Flower color",
    "Flower sex",
    "Fruit type",
    "Plant growth form",
    "Plant lifespan (longevity)",
    "Seed germination requirement"
]

# Melt the DataFrame to convert trait columns into 'TraitName' and 'TraitValue' columns
try_df_melted = try_df.melt(id_vars=["SpeciesName"], var_name="TraitName", value_name="TraitValue")

# Now filter the melted DataFrame by 'TraitName'
try_df = try_df_melted[try_df_melted["TraitName"].isin(selected_traits)]

In [ ]:
try_wide = try_df.pivot_table(
    index="SpeciesName",
    columns="TraitName",
    values="TraitValue",
    aggfunc="first" # Changed aggfunc from 'mean' to 'first' as 'mean' might not be appropriate for all 'TraitValue' types
).reset_index()

print(try_wide.head())

TraitName          SpeciesName Flower color Flower sex Fruit type  \
0            acacia abbreviata          NaN        NaN        NaN   
1               acacia abrupta          NaN        NaN        NaN   
2           acacia acanthaster          NaN        NaN        NaN   
3          acacia acanthoclada          NaN        NaN        NaN   
4            acacia acellerata          NaN        NaN        NaN   

TraitName Plant growth form Plant lifespan (longevity)  \
0                     Shrub                        NaN   
1             Free-standing                        NaN   
2             Free-standing                        NaN   
3               shrub/woody                        NaN   
4             Free-standing                        NaN   

TraitName Seed germination requirement  
0                                  NaN  
1                                  NaN  
2                                  NaN  
3                                  NaN  
4                              

In [ ]:
import requests
import pandas as pd
import time
import re

df = pd.read_csv("wfo_species_clean.csv")
species_list = df["scientificName"].dropna().unique()


# -------------------------
# CLEAN NAME (CRITICAL FIX)
# -------------------------
def clean_name(name):
    name = str(name).lower()
    name = name.split("(")[0]
    name = name.replace("l.", "").replace("linn.", "")
    parts = name.split()
    return " ".join(parts[:2])


# -------------------------
# WIKIPEDIA FULL TEXT API
# -------------------------
def get_wiki_text(species):

    species = clean_name(species)

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "titles": species,
        "explaintext": True
    }

    try:
        r = requests.get(url, params=params, timeout=10)
        data = r.json()

        page = next(iter(data["query"]["pages"].values()))

        return page.get("extract", None)

    except:
        return None


# -------------------------
# TRAIT EXTRACTION (ROBUST)
# -------------------------
def extract_traits(text):

    if not text:
        return {}

    text_lower = text.lower()

    traits = {}

    # ---------------- NATIVE ----------------
    native_patterns = [
        r"native to ([^.]+)",
        r"native throughout ([^.]+)",
        r"found in ([^.]+)",
        r"occurs in ([^.]+)"
    ]

    for p in native_patterns:
        m = re.search(p, text_lower)
        if m:
            traits["native_range"] = m.group(1)
            break

    # ---------------- CLIMATE ----------------
    if any(x in text_lower for x in ["tropical", "subtropical"]):
        traits["climate"] = "tropical"
    elif "temperate" in text_lower:
        traits["climate"] = "temperate"

    # ---------------- FLOWERING INFO ----------------
    if any(x in text_lower for x in ["flower", "bloom", "flowering"]):
        traits["has_flowering_info"] = True

    return traits


# -------------------------
# RUN PIPELINE
# -------------------------
results = []

for i, sp in enumerate(species_list[:8000]):  # test first

    text = get_wiki_text(sp)
    traits = extract_traits(text)

    results.append({
        "scientificName": clean_name(sp),
        "has_text": text is not None,
        **traits
    })

    if i % 50 == 0:
        print(f"Processed {i}")

    time.sleep(0.2)


df_traits = pd.DataFrame(results)
df_traits.to_csv("species_traits.csv", index=False)

print("DONE")

Processed 0
Processed 50
Processed 100
Processed 150
Processed 200
Processed 250
Processed 300
Processed 350
Processed 400
Processed 450
Processed 500
Processed 550
Processed 600
Processed 650
Processed 700
Processed 750
Processed 800
Processed 850
Processed 900
Processed 950
Processed 1000
Processed 1050
Processed 1100
Processed 1150
Processed 1200
Processed 1250
Processed 1300
Processed 1350
Processed 1400
Processed 1450
Processed 1500
Processed 1550
Processed 1600
Processed 1650
Processed 1700
Processed 1750
Processed 1800
Processed 1850
Processed 1900
Processed 1950
Processed 2000
Processed 2050
Processed 2100
Processed 2150
Processed 2200
Processed 2250
Processed 2300
Processed 2350
Processed 2400
Processed 2450
Processed 2500
Processed 2550
Processed 2600
Processed 2650
Processed 2700
Processed 2750
Processed 2800
Processed 2850
Processed 2900
Processed 2950
Processed 3000
Processed 3050
Processed 3100
Processed 3150
Processed 3200
Processed 3250
Processed 3300
Processed 3350
Pro

In [ ]:
wiki_text = get_wiki_text("Ficus religiosa")
if wiki_text:
    print(wiki_text[:1000])
else:
    print("Could not retrieve text for Ficus religiosa.")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
wiki_df = pd.DataFrame(wiki_results)

In [ ]:
print(wiki_df.columns)

Index(['scientificName', 'native_range', 'climate', 'bloom_time',
       'flower_color'],
      dtype='object')


In [ ]:
print(get_wiki_traits_loose("bauhinia variegata"))

{'scientificName': 'bauhinia variegata', 'native_flag': False, 'climate_flag': False, 'flower_flag': False, 'color_flag': False, 'native_text': None, 'climate_text': None, 'flower_text': None, 'color_text': None}


In [ ]:
species_list = try_wide["SpeciesName"].dropna().unique()

wiki_results = []

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(get_wiki_traits, sp): sp for sp in species_list[:3000]}

    for i, future in enumerate(as_completed(futures)):
        wiki_results.append(future.result())

        if i % 100 == 0:
            print(f"Processed {i}")

wiki_df = pd.DataFrame(wiki_results)

Processed 0
Processed 100
Processed 200
Processed 300
Processed 400
Processed 500
Processed 600
Processed 700
Processed 800
Processed 900
Processed 1000
Processed 1100
Processed 1200
Processed 1300
Processed 1400
Processed 1500
Processed 1600
Processed 1700
Processed 1800
Processed 1900
Processed 2000
Processed 2100
Processed 2200
Processed 2300
Processed 2400
Processed 2500
Processed 2600
Processed 2700
Processed 2800
Processed 2900


In [ ]:
wiki_df = wiki_df.rename(columns={"scientificName": "SpeciesName"})
final_df = try_wide.merge(wiki_df, on="SpeciesName", how="left")

final_df.to_csv("final_tree_traits.csv", index=False)

print(final_df.head())

            SpeciesName Flower color Flower sex Fruit type Plant growth form  \
0    acacia acrionastes          NaN        NaN        NaN        tree/woody   
1      acacia alleniana          NaN        NaN        NaN        tree/woody   
2        acacia ammobia          NaN        NaN        NaN              tree   
3         acacia amoena          NaN        NaN        NaN        tree/woody   
4  acacia ancistrocarpa          NaN        NaN        NaN        tree/woody   

  Plant lifespan (longevity) Seed germination requirement  
0                        NaN                          NaN  
1                        NaN                          NaN  
2                        NaN                          NaN  
3                        NaN                          NaN  
4                        NaN                          NaN  


In [ ]:
def clean_name(x):
    if pd.isna(x):
        return x
    return x.strip().lower()

try_wide["SpeciesName"] = try_wide["SpeciesName"].apply(clean_name)
wiki_df["SpeciesName"] = wiki_df["SpeciesName"].apply(clean_name)

In [ ]:
print("TRY species:", try_wide["SpeciesName"].nunique())
print("WIKI species:", wiki_df["SpeciesName"].nunique())

common = set(try_wide["SpeciesName"]) & set(wiki_df["SpeciesName"])
print("Matching species:", len(common))

TRY species: 3050
WIKI species: 3000
Matching species: 3000


In [ ]:
final_df = try_wide.merge(
    wiki_df,
    on="SpeciesName",
    how="left"
)

In [ ]:
final_df.to_csv("final_tree_traits.csv", index=False)

In [ ]:
print(final_df.head())

print("Total species:", final_df["SpeciesName"].nunique())

# Check for column existence before accessing
if "native_range" in final_df.columns:
    print("With native range:", final_df["native_range"].notna().sum())
else:
    print("Column 'native_range' is missing in final_df.")

if "climate" in final_df.columns:
    print("With climate:", final_df["climate"].notna().sum())
else:
    print("Column 'climate' is missing in final_df.")

if "bloom_time" in final_df.columns:
    print("With bloom time:", final_df["bloom_time"].notna().sum())
else:
    print("Column 'bloom_time' is missing in final_df.")

            SpeciesName Flower color Flower sex Fruit type Plant growth form  \
0    acacia acrionastes          NaN        NaN        NaN        tree/woody   
1      acacia alleniana          NaN        NaN        NaN        tree/woody   
2        acacia ammobia          NaN        NaN        NaN              tree   
3         acacia amoena          NaN        NaN        NaN        tree/woody   
4  acacia ancistrocarpa          NaN        NaN        NaN        tree/woody   

  Plant lifespan (longevity) Seed germination requirement  
0                        NaN                          NaN  
1                        NaN                          NaN  
2                        NaN                          NaN  
3                        NaN                          NaN  
4                        NaN                          NaN  
Total species: 3050
Column 'native_range' is missing in final_df.
Column 'climate' is missing in final_df.
Column 'bloom_time' is missing in final_df.


merging flowering plant file

In [1]:
import pandas as pd
import glob
import os

# Folder containing your CSV files
folder_path = r"/content/drive/MyDrive/flowering plant families"   # <-- change this

# Get all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

print(f"Found {len(csv_files)} CSV files.")

# -----------------------------
# STEP 1: CHECK COLUMN MATCHING
# -----------------------------
column_sets = {}
all_match = True

for file in csv_files:
    try:
        df = pd.read_csv(file, nrows=0, encoding='latin1')   # read header only, with specified encoding
        column_sets[file] = list(df.columns)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        all_match = False

# Handle case where csv_files might be empty or all failed to read
if not csv_files or not column_sets:
    print("No CSV files found or could not read any headers. Exiting.")
    all_match = False
    # Optionally, you might want to exit here or handle more gracefully
    # For now, we'll let the 'if all_match' block handle it.

if all_match and csv_files:
    reference_file = csv_files[0]
    reference_cols = column_sets[reference_file]

    for file, cols in column_sets.items():
        if cols != reference_cols:
            all_match = False
            print(f"\nColumn mismatch in: {os.path.basename(file)}")
            print("Expected:", reference_cols)
            print("Found   :", cols)

# -----------------------------
# STEP 2: MERGE + FILTER SPECIES
# -----------------------------
if all_match:
    print("\nAll column names match. Reading and merging files...")

    dataframes = []

    for file in csv_files:
        try:
            df = pd.read_csv(file, encoding='latin1') # Read full file with specified encoding

            # Keep only rows where taxonRank = species
            if "taxonRank" in df.columns:
                df = df[df["taxonRank"].astype(str).str.lower() == "species"]

            dataframes.append(df)
        except Exception as e:
            print(f"Error reading {file} for merging: {e}")

    if dataframes:
        merged_df = pd.concat(dataframes, ignore_index=True)

        output_file = os.path.join(folder_path, "merged_species_only.csv")
        merged_df.to_csv(output_file, index=False)

        print(f"Merged {len(csv_files)} files successfully.")
        print(f"Final rows (species only): {len(merged_df)}")
        print(f"Saved to: {output_file}")
    else:
        print("No dataframes were successfully loaded for merging.")

else:
    print("\nMerge stopped because some column names do not match or an error occurred during initial file reading.")


Found 418 CSV files.

All column names match. Reading and merging files...


/tmp/ipykernel_12736/2581629097.py:55: DtypeWarning: Columns (5,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='latin1') # Read full file with specified encoding
/tmp/ipykernel_12736/2581629097.py:55: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='latin1') # Read full file with specified encoding
/tmp/ipykernel_12736/2581629097.py:55: DtypeWarning: Columns (5,8,9,10,12,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='latin1') # Read full file with specified encoding
/tmp/ipykernel_12736/2581629097.py:55: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='latin1') # Read full file with specified encoding
/tmp/ipykernel_12736/2581629097.py:55: DtypeWarning: Columns (5,10,19) have mixed

Merged 418 files successfully.
Final rows (species only): 1040697
Saved to: /content/drive/MyDrive/flowering plant families/merged_species_only.csv


In [4]:
import pandas as pd
import glob
import os

# =====================================================
# CHANGE THIS TO YOUR FOLDER PATH
# =====================================================
folder_path = r"/content/drive/MyDrive/flowering plant families"

# =====================================================
# GET ALL CSV FILES
# =====================================================
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

print(f"Found {len(csv_files)} CSV files.")

if len(csv_files) == 0:
    raise ValueError("No CSV files found in folder.")

# =====================================================
# STEP 1: CHECK COLUMN MATCHING
# =====================================================
column_sets = {}
all_match = True

for file in csv_files:
    try:
        df_header = pd.read_csv(file, nrows=0, encoding="latin1")
        column_sets[file] = list(df_header.columns)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        all_match = False

reference_file = csv_files[0]
reference_cols = column_sets[reference_file]

for file, cols in column_sets.items():
    if cols != reference_cols:
        all_match = False
        print(f"\nColumn mismatch in: {os.path.basename(file)}")
        print("Expected:", reference_cols)
        print("Found   :", cols)

if not all_match:
    print("\nMerge stopped because some column names do not match.")
    raise SystemExit()

print("\nAll column names match. Reading and filtering files...")

# =====================================================
# STEP 2: PROCESS FILES
# =====================================================
all_data = []

allowed_nomenclature = ["valid", "conserved", ""]

for i, file in enumerate(csv_files, start=1):
    print(f"Processing {i}/{len(csv_files)} : {os.path.basename(file)}")

    try:
        df = pd.read_csv(file, encoding="latin1", low_memory=False)

        # -------------------------
        # Species only
        # -------------------------
        df["taxonRank"] = df["taxonRank"].astype(str).str.strip().str.lower()
        df = df[df["taxonRank"] == "species"]

        # -------------------------
        # Accepted only
        # -------------------------
        df["taxonomicStatus"] = df["taxonomicStatus"].astype(str).str.strip().str.lower()
        df = df[df["taxonomicStatus"] == "accepted"]

        # -------------------------
        # Valid nomenclature only
        # -------------------------
        df["nomenclaturalStatus"] = (
            df["nomenclaturalStatus"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
        )

        df = df[df["nomenclaturalStatus"].isin(allowed_nomenclature)]

        all_data.append(df)

    except Exception as e:
        print(f"Skipped {file} due to error: {e}")

# =====================================================
# STEP 3: MERGE
# =====================================================
merged_df = pd.concat(all_data, ignore_index=True)

print(f"\nRows after filtering: {len(merged_df):,}")

# =====================================================
# STEP 4: REMOVE DUPLICATES
# =====================================================
if "scientificName" in merged_df.columns:
    before = len(merged_df)
    merged_df = merged_df.drop_duplicates(subset="scientificName")
    after = len(merged_df)

    print(f"Duplicates removed: {before - after:,}")
    print(f"Final rows: {after:,}")

# =====================================================
# STEP 5: SAVE OUTPUT
# =====================================================
output_file = os.path.join(folder_path, "flowering_plant_families.csv")
merged_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"\nSaved successfully:")
print(output_file)

Found 419 CSV files.

All column names match. Reading and filtering files...
Processing 1/419 : Eupteleaceae.csv
Processing 2/419 : Papaveraceae.csv
Processing 3/419 : Circaeasteraceae.csv
Processing 4/419 : Lardizabalaceae.csv
Processing 5/419 : Menispermaceae.csv
Processing 6/419 : Berberidaceae.csv
Processing 7/419 : Ranunculaceae.csv
Processing 8/419 : Nelumbonaceae.csv
Processing 9/419 : Platanaceae.csv
Processing 10/419 : Proteaceae.csv
Processing 11/419 : Sabiaceae.csv
Processing 12/419 : Trochodendraceae.csv
Processing 13/419 : Buxaceae.csv
Processing 14/419 : Gunneraceae.csv
Processing 15/419 : Myrothamnaceae.csv
Processing 16/419 : Dilleniaceae.csv
Processing 17/419 : Altingiaceae.csv
Processing 18/419 : Aphanopetalaceae.csv
Processing 19/419 : Cercidiphyllaceae.csv
Processing 20/419 : Crassulaceae.csv
Processing 21/419 : Cynomoriaceae.csv
Processing 22/419 : Daphniphyllaceae.csv
Processing 23/419 : Grossulariaceae.csv
Processing 24/419 : Haloragaceae.csv
Processing 25/419 : 